In [ ]:
import ctypes
import math
import numpy as np
import glfw
from OpenGL.GL import *
from OpenGL.GL.shaders import compileProgram, compileShader
from PIL import Image

# ==============================================================================
# CONFIGURATION: Update filenames to match your downloaded/converted assets
# ==============================================================================
OBJ_FILE = "model.obj"
TEX_FILE = "texture.jpg"
WINDOW_WIDTH, WINDOW_HEIGHT = 900, 700


# ==============================================================================
# 1. WAVEFRONT OBJ PARSER & NORMALIZATION (25 Points)
# ==============================================================================
def unpack_face_vertex(token, raw_v, raw_vt, raw_vn, out_v, out_vt, out_vn):
    """
    Unpacks a single 'v/vt/vn' face token (1-based indices) and appends
    the corresponding 0-based vertex position, UV, and normal to output lists.
    """
    indices = token.split('/')

    # TODO 1A: Parse 1-based vertex index (indices[0]), convert to 0-based,
    # and append raw_v[v_idx] to out_v.
    pass

    # TODO 1B: Check if texture coordinate index exists (len(indices) > 1 and indices[1]).
    # If present, convert to 0-based index and append from raw_vt; otherwise append [0.0, 0.0].
    pass

    # TODO 1C: Check if vertex normal index exists (len(indices) > 2 and indices[2]).
    # If present, convert to 0-based index and append from raw_vn; otherwise append [0.0, 1.0, 0.0].
    pass


def load_obj(filename):
    """
    Parses an ASCII Wavefront .obj file into contiguous float32 NumPy arrays
    of triangle vertices, UVs, and vertex normals.
    """
    raw_v, raw_vt, raw_vn = [], [], []
    vertices, uvs, normals = [], [], []

    with open(filename, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            parts = line.strip().split()
            if not parts:
                continue

            tag = parts[0]

            # TODO 1D: Handle tags 'v', 'vt', 'vn', and 'f':
            # - 'v': append [float(x), float(y), float(z)] to raw_v
            # - 'vt': append [float(u), float(v)] to raw_vt
            # - 'vn': append [float(nx), float(ny), float(nz)] to raw_vn
            # - 'f': triangulate polygons! Always emit triangle [0, 1, 2].
            #        If len(face_tokens) == 4 (quad), also emit triangle [0, 2, 3].
            pass

    return (
        np.array(vertices, dtype=np.float32),
        np.array(uvs, dtype=np.float32),
        np.array(normals, dtype=np.float32),
    )


def standardize_mesh(vertices):
    """
    Translates the mesh center to (0, 0, 0) and scales it uniformly so that
    its largest dimension fits cleanly inside [-0.8, 0.8]^3.
    """
    # TODO 2: Compute min_pt, max_pt, bounding box center, and max_extent.
    # Translate vertices so center is at origin, then scale by (1.6 / max_extent).
    return vertices


# ==============================================================================
# 2. GLSL SHADERS FOR SMOOTH PHONG SHADING (25 Points)
# ==============================================================================
VERTEX_SHADER_SRC = """
#version 330 core
layout (location = 0) in vec3 aPos;
layout (location = 1) in vec2 aUv;
layout (location = 2) in vec3 aNormal;

uniform mat4 uModelView;
uniform mat4 uPersp;

out vec3 vPoint;
out vec2 vUv;
out vec3 vNormal;

void main() {
    // TODO 3A: Transform vertex position into eye/camera space (vPoint)
    // and compute clip-space gl_Position using uPersp.
    gl_Position = vec4(aPos, 1.0); // Replace this placeholder
    vPoint = aPos;
    vUv = aUv;

    // TODO 3B: Transform aNormal into eye space using uModelView with w = 0.0
    // so translation does not affect direction vectors, then normalize.
    vNormal = aNormal; // Replace this placeholder
}
"""

FRAGMENT_SHADER_SRC = """
#version 330 core
in vec3 vPoint;
in vec2 vUv;
in vec3 vNormal;

out vec4 FragColor;

uniform sampler2D uTexture;
uniform int uNumLights;
uniform vec3 uLights[8];
uniform float uAmb = 0.25;
uniform float uDif = 0.70;
uniform float uSpc = 0.40;

void main() {
    // TODO 3C: Renormalize interpolated vNormal and compute view direction E = normalize(-vPoint).
    // Loop over uNumLights to accumulate diffuse (max(0, dot(N, L))) and specular
    // (pow(max(0, dot(R, E)), 32.0)) contributions.
    // Sample uTexture at vUv and combine lighting with texColor.
    vec3 texColor = texture(uTexture, vUv).rgb;
    FragColor = vec4(texColor, 1.0); // Replace with shaded output
}
"""


# ==============================================================================
# 3. CAMERA MATH & INTERACTIVE MOUSE CONTROLS (Provided Boilerplate)
# ==============================================================================
class CameraState:
    def __init__(self):
        self.rot_x = 15.0
        self.rot_y = -30.0
        self.distance = 2.6
        self.dragging = False
        self.last_x = 0.0
        self.last_y = 0.0


def perspective_matrix(fov_deg, aspect, z_near, z_far):
    f = 1.0 / math.tan(math.radians(fov_deg) / 2.0)
    M = np.zeros((4, 4), dtype=np.float32)
    M[0, 0] = f / aspect
    M[1, 1] = f
    M[2, 2] = (z_far + z_near) / (z_near - z_far)
    M[2, 3] = (2.0 * z_far * z_near) / (z_near - z_far)
    M[3, 2] = -1.0
    return M


def modelview_matrix(cam):
    rx = math.radians(cam.rot_x)
    ry = math.radians(cam.rot_y)

    Rx = np.array([
        [1, 0, 0, 0],
        [0, math.cos(rx), -math.sin(rx), 0],
        [0, math.sin(rx), math.cos(rx), 0],
        [0, 0, 0, 1]
    ], dtype=np.float32)

    Ry = np.array([
        [math.cos(ry), 0, math.sin(ry), 0],
        [0, 1, 0, 0],
        [-math.sin(ry), 0, math.cos(ry), 0],
        [0, 0, 0, 1]
    ], dtype=np.float32)

    T = np.eye(4, dtype=np.float32)
    T[2, 3] = -cam.distance
    return T @ Rx @ Ry


def load_texture(filepath):
    """Loads an image via Pillow, flips vertically for OpenGL UV convention, and uploads."""
    img = Image.open(filepath).convert("RGB").transpose(Image.FLIP_TOP_BOTTOM)
    img_data = np.array(img, dtype=np.uint8)

    tex_id = glGenTextures(1)
    glBindTexture(GL_TEXTURE_2D, tex_id)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR_MIPMAP_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)

    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGB, img.width, img.height, 0, GL_RGB, GL_UNSIGNED_BYTE, img_data)
    glGenerateMipmap(GL_TEXTURE_2D)
    glBindTexture(GL_TEXTURE_2D, 0)
    return tex_id


# ==============================================================================
# 4. MAIN RENDERING APPLICATION
# ==============================================================================
def main():
    if not glfw.init():
        raise RuntimeError("Failed to initialize GLFW")

    glfw.window_hint(glfw.CONTEXT_VERSION_MAJOR, 3)
    glfw.window_hint(glfw.CONTEXT_VERSION_MINOR, 3)
    glfw.window_hint(glfw.OPENGL_PROFILE, glfw.OPENGL_CORE_PROFILE)

    window = glfw.create_window(WINDOW_WIDTH, WINDOW_HEIGHT, "HW6: Smooth Shaded OBJ Mesh", None, None)
    if not window:
        glfw.terminate()
        raise RuntimeError("Failed to create GLFW window")

    glfw.make_context_current(window)
    glEnable(GL_DEPTH_TEST)

    # Set up mouse callbacks for 3D inspection
    cam = CameraState()

    def mouse_button_cb(win, button, action, mods):
        if button == glfw.MOUSE_BUTTON_LEFT:
            cam.dragging = (action == glfw.PRESS)
            cam.last_x, cam.last_y = glfw.get_cursor_pos(win)

    def cursor_pos_cb(win, xpos, ypos):
        if cam.dragging:
            dx = xpos - cam.last_x
            dy = ypos - cam.last_y
            cam.rot_y += dx * 0.4
            cam.rot_x += dy * 0.4
            cam.last_x, cam.last_y = xpos, ypos

    def scroll_cb(win, xoffset, yoffset):
        cam.distance = max(0.8, min(10.0, cam.distance - yoffset * 0.2))

    glfw.set_mouse_button_callback(window, mouse_button_cb)
    glfw.set_cursor_pos_callback(window, cursor_pos_cb)
    glfw.set_scroll_callback(window, scroll_cb)

    # Compile shaders
    program = compileProgram(
        compileShader(VERTEX_SHADER_SRC, GL_VERTEX_SHADER),
        compileShader(FRAGMENT_SHADER_SRC, GL_FRAGMENT_SHADER),
    )

    # Load and standardize mesh data
    vertices, uvs, normals = load_obj(OBJ_FILE)
    if len(vertices) == 0:
        raise RuntimeError("No vertices loaded! Complete TODO 1A-1D first.")
    vertices = standardize_mesh(vertices)

    # ==========================================================================
    # TODO 4: GPU BUFFER ALLOCATION & VAO ATTRIBUTE LAYOUT (20 Points)
    # 1. Horizontally stack vertices, uvs, and normals into a single interleaved
    #    float32 array: [x, y, z, u, v, nx, ny, nz] (8 floats = 32 bytes/vertex).
    # 2. Generate and bind a VAO and VBO.
    # 3. Upload the interleaved buffer to GL_ARRAY_BUFFER using GL_STATIC_DRAW.
    # 4. Configure glVertexAttribPointer & glEnableVertexAttribArray for:
    #    - location 0: position (3 floats, stride=32, offset=0)
    #    - location 1: uv       (2 floats, stride=32, offset=12 bytes)
    #    - location 2: normal   (3 floats, stride=32, offset=20 bytes)
    # ==========================================================================
    vao = glGenVertexArrays(1)
    vbo = glGenBuffers(1)
    # Write VBO upload and attribute pointer setup here...

    # Load 2D texture
    texture_id = load_texture(TEX_FILE)

    # Query uniform locations
    u_mv_loc = glGetUniformLocation(program, "uModelView")
    u_persp_loc = glGetUniformLocation(program, "uPersp")
    u_num_lights_loc = glGetUniformLocation(program, "uNumLights")
    u_lights_loc = glGetUniformLocation(program, "uLights")
    u_tex_loc = glGetUniformLocation(program, "uTexture")

    # Eye-space light positions
    lights = np.array([
        [2.5, 3.0, 3.0],
        [-2.5, -1.5, 2.0],
    ], dtype=np.float32)

    while not glfw.window_should_close(window):
        glfw.poll_events()
        fb_w, fb_h = glfw.get_framebuffer_size(window)
        glViewport(0, 0, fb_w, fb_h)

        glClearColor(0.12, 0.14, 0.18, 1.0)
        glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

        glUseProgram(program)

        # Upload camera matrices (transpose=GL_TRUE because NumPy is row-major)
        aspect = fb_w / max(1, fb_h)
        persp = perspective_matrix(45.0, aspect, 0.1, 100.0)
        mv = modelview_matrix(cam)
        glUniformMatrix4fv(u_persp_loc, 1, GL_TRUE, persp)
        glUniformMatrix4fv(u_mv_loc, 1, GL_TRUE, mv)

        # Upload lighting uniforms
        glUniform1i(u_num_lights_loc, len(lights))
        glUniform3fv(u_lights_loc, len(lights), lights)

        # Bind texture unit 0
        glActiveTexture(GL_TEXTURE0)
        glBindTexture(GL_TEXTURE_2D, texture_id)
        glUniform1i(u_tex_loc, 0)

        # Draw mesh
        glBindVertexArray(vao)
        glDrawArrays(GL_TRIANGLES, 0, len(vertices))
        glBindVertexArray(0)

        glfw.swap_buffers(window)

    glDeleteVertexArrays(1, [vao])
    glDeleteBuffers(1, [vbo])
    glDeleteTextures(1, [texture_id])
    glfw.terminate()


if __name__ == "__main__":
    main()

ShaderValidationError: Validation failure (np.int32(0)): b'Validation Failed: No vertex array object bound.\n'

: 